# 10. RAG の検索をよくする

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の実験 10 です。

[実験08](../docs/results/08_rag_api.md) の RAG は正答 **64%**、検索が正解の段落を拾えたのは **76%** でした。外した 9 問のうち 6 問は **検索ミス** です。
原因は 2 つ見えていました。

- どの質問にも「colab-oss-lab」が入っていて、README が毎回上位に来てしまう
- 埋め込みモデル（`multilingual-e5-small`）が小さい

ここでは **モデル・資料・質問は 08 と同じ** にして、検索のやり方だけを 1 段ずつ変えます。

| 設定 | 変えること |
|---|---|
| A | 08 と同じ（e5-small、上位 5 段落） |
| B | 質問から「colab-oss-lab」などのリポジトリ名を外して検索する |
| C | B ＋ 大きい埋め込みモデル（`multilingual-e5-large`） |
| D | C ＋ ハイブリッド検索（言葉の一致で探す BM25 と、意味で探す埋め込みを合わせる） |
| E | D ＋ リランカー（上位 30 段落を `bge-reranker-v2-m3` で並べ直す） |
| F | E ＋ 質問の言い換え（モデルに検索語を 3 通り作らせてから探す） |

- モデル: `cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit`（vLLM の OpenAI 互換 API、08 と同じ）
- 資料: 08 を実行したときのリポジトリ（コミット `e4570f0` に固定）の 14 ファイル → 147 段落
- 質問: 06・08 と同じ 25 問 ＋ 資料にない 5 問

「想定どおり」とは（いちばん良い設定で）:

- 検索が正解の段落を拾えた割合（上位 5 段落）が **90% 以上**（08 は 76%）
- RAG の正答率が **80% 以上**（08 は 64%）
- 資料にない質問 5 問のうち 4 問以上で「記載がありません」と答える（作り話が増えない）
- 1 問あたりの検索が 1 秒以内

所要時間の目安: 20〜30 分。

---

## 実行する前に

1. **ランタイム → ランタイムのタイプを変更 → L4 GPU → Save**
2. 上から順に ▶（または「すべてのセルを実行」）
3. 終わったら **ランタイム → セッションを管理 → 解放**

## 1. GPU を確認して、ライブラリを入れる

In [ ]:
import subprocess
q = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True)
gpu_name, mem_mib = [x.strip() for x in q.strip().split(",")]
vram_total_gb = int(mem_mib) / 1024
assert "L4" in gpu_name, f"GPU が L4 ではありません: {gpu_name}"
print("GPU:", gpu_name, round(vram_total_gb, 1), "GB")

In [ ]:
# vLLM は専用の仮想環境に入れる（実験04・08・09 と同じ）
!pip install -q uv
!uv venv -q --allow-existing /content/vllm-env
!uv pip install --python /content/vllm-env/bin/python vllm 2>&1 | tail -2
# ノート本体（検索と API の呼び出し）用
!pip install -q -U openai sentence-transformers rank_bm25 2>&1 | tail -2
!/content/vllm-env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

## 2. vLLM の API サーバーを起動する

08 との違いは `--gpu-memory-utilization` を 0.92 → **0.90** にしたことだけです。
空いた GPU（約 2GB）にリランカー（約 1.1GB）を載せます。大きい埋め込みモデルは CPU で動かします。
**サーバーを先に起動してから** 検索のモデルを載せます（逆だと vLLM が必要な空きを確保できない）。

- 最初は 0.80 にしたが、`ValueError: To serve at least one request with the model's max seq len (8192) ...` で起動が止まった（重みと作業用メモリで 0.80 をほぼ使い切り、会話の記憶が 8,192 トークン分も残らない）

In [ ]:
import os, re, time, requests

MODEL = "cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit"
server_log = open("/content/vllm_server.log", "w")
t0 = time.time()
server = subprocess.Popen(
    ["/content/vllm-env/bin/vllm", "serve", MODEL,
     "--served-model-name", "gemma4-26b",
     "--language-model-only",
     "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.90",
     "--kv-cache-dtype", "fp8",
     "--max-num-batched-tokens", "2048",
     "--max-num-seqs", "16",
     "--port", "8000"],
    stdout=server_log, stderr=subprocess.STDOUT,
    # 仮想環境の bin を PATH に入れる（入れないと起動の途中で FileNotFoundError: 'ninja'）
    env={**os.environ, "PATH": "/content/vllm-env/bin:" + os.environ["PATH"]})

while True:
    if server.poll() is not None:
        causes = [l for l in open("/content/vllm_server.log").read().splitlines()
                  if re.search(r"(Error|error:|out of memory)", l) and "File " not in l]
        print("\n".join(causes[-8:]))
        raise RuntimeError("vLLM サーバーが止まりました（上の原因の行、または /content/vllm_server.log を確認）")
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(5)
server_start_min = (time.time() - t0) / 60
kv = re.findall(r"GPU KV cache size: ([\d,]+) tokens", open("/content/vllm_server.log").read())
print(f"サーバー起動: {server_start_min:.1f} 分 / KV キャッシュ: {kv[-1] if kv else '-'} トークン")

## 3. 資料を集めて、4 種類の検索を用意する

資料は 08 を実行したときと同じもの（コミット `e4570f0`）を取ってきて、08 と同じ分け方で 147 段落にします。

In [ ]:
import json, urllib.request
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

RAW = "https://raw.githubusercontent.com/moruku36/colab-oss-lab/e4570f0/"   # 08 を実行したときのリポジトリ
FILES = ["README.md",
         "docs/01-what-is-colab.md", "docs/02-google-ai-pro.md", "docs/03-what-you-can-do.md",
         "docs/04-oss-models.md", "docs/05-gpu-basics.md", "docs/06-quantization.md", "docs/glossary.md",
         "docs/results/01_first_run.md", "docs/results/02_quantization_max.md", "docs/results/03_thinking_on_off.md",
         "docs/results/04_vllm_speedup.md", "docs/results/05_quantization_compare.md", "docs/results/07_l4_limit.md"]

def chunks_of(path, text, size=600):   # 08 と同じ分け方
    out, head, buf = [], "", ""
    for line in text.splitlines():
        if line.startswith("#"):
            if buf.strip():
                out.append((path, head, buf.strip()))
            head, buf = line.lstrip("# ").strip(), ""
            continue
        if len(buf) + len(line) > size and buf.strip():
            out.append((path, head, buf.strip()))
            buf = ""
        buf += line + "\n"
    if buf.strip():
        out.append((path, head, buf.strip()))
    return out

chunks = []
for f in FILES:
    chunks += chunks_of(f, urllib.request.urlopen(RAW + f).read().decode("utf-8"))
texts = [f"{p} / {h}\n{b}" for p, h, b in chunks]
print("段落の数:", len(chunks))

# 意味で探す（埋め込み）: 小さい（08 と同じ）も大きいも CPU（GPU は vLLM がほぼ使っている）
t = time.time()
emb_small = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
emb_large = SentenceTransformer("intfloat/multilingual-e5-large", device="cpu")
P = {"small": emb_small.encode([f"passage: {t}" for t in texts], normalize_embeddings=True, batch_size=32),
     "large": emb_large.encode([f"passage: {t}" for t in texts], normalize_embeddings=True, batch_size=32)}
EMB = {"small": emb_small, "large": emb_large}
print(f"埋め込み（147 段落 × 2 モデル、CPU）: {time.time() - t:.0f} 秒")

# 言葉の一致で探す（BM25）: 英数字は単語、日本語は 2 文字ずつに区切る
def tokenize(s):
    s = s.lower()
    toks = re.findall(r"[a-z0-9][a-z0-9.\-]*", s)
    for run in re.findall(r"[^\sa-z0-9.\-、。，．・（）()「」『』【】\[\]|:：/\\#*`<>=!?！？]+", s):
        toks += [run[i:i + 2] for i in range(max(1, len(run) - 1))]
    return toks
bm25 = BM25Okapi([tokenize(t) for t in texts])

# 並べ直す（リランカー）: GPU の空きに載せる。載らなければ CPU
try:
    reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=512, model_kwargs={"torch_dtype": torch.float16})
    reranker.predict([("テスト", "テスト " * 200)] * 8, batch_size=8, show_progress_bar=False)
    rerank_device = "GPU（fp16）"
except torch.cuda.OutOfMemoryError:
    reranker = None; torch.cuda.empty_cache()
    reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cpu", max_length=512)
    rerank_device = "CPU（GPU に載らなかった）"
print("リランカー:", rerank_device)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

# リポジトリ名だけを外す（「たとえで」「手順で」などは残す）
REPO_WORDS = [r"（colab-oss-lab の結果）", r"colab-oss-lab (によると|では|で|の)、?\s*", r"colab-oss-lab", r"このリポジトリの"]
def clean(q):
    for w in REPO_WORDS:
        q = re.sub(w, "", q)
    return q.strip("、 ") or q

def dense(q, which, n):
    v = EMB[which].encode([f"query: {q}"], normalize_embeddings=True)[0]
    return list(np.argsort(-(P[which] @ v))[:n])

def sparse(q, n):
    return list(np.argsort(-bm25.get_scores(tokenize(q)))[:n])

def rrf(lists, k=60):   # 複数の順位を合わせる（Reciprocal Rank Fusion）
    score = {}
    for l in lists:
        for r, i in enumerate(l):
            score[i] = score.get(i, 0) + 1 / (k + r + 1)
    return sorted(score, key=lambda i: -score[i])

def rerank(q, cand):
    s = reranker.predict([(q, texts[i]) for i in cand], batch_size=8, show_progress_bar=False)
    return [cand[j] for j in np.argsort(-s)]

REWRITE = ("次の質問に答えるための資料を検索します。検索に使う短い日本語の検索語を 3 通り作ってください。"
           "言い換えや、答えが書かれていそうな言葉を入れてください。リポジトリ名（colab-oss-lab）は入れないでください。"
           "1 行に 1 つ、番号や説明はつけないでください。\n質問: ")
_rw = {}
def rewrite(q):
    if q not in _rw:
        r = client.chat.completions.create(model="gemma4-26b", messages=[{"role": "user", "content": REWRITE + q}],
                                           max_tokens=80, temperature=0, extra_body=NO_THINK)
        _rw[q] = [l.strip("-・* 0123456789.") for l in r.choices[0].message.content.splitlines() if l.strip()][:3]
    return _rw[q]

CONFIGS = {
    "A": dict(name="08 と同じ（e5-small）", clean=False, emb="small", hybrid=False, rerank=False, rewrite=False),
    "B": dict(name="質問からリポジトリ名を外す", clean=True, emb="small", hybrid=False, rerank=False, rewrite=False),
    "C": dict(name="B + 大きい埋め込み（e5-large）", clean=True, emb="large", hybrid=False, rerank=False, rewrite=False),
    "D": dict(name="C + ハイブリッド（BM25）", clean=True, emb="large", hybrid=True, rerank=False, rewrite=False),
    "E": dict(name="D + リランカー", clean=True, emb="large", hybrid=True, rerank=True, rewrite=False),
    "F": dict(name="E + 質問の言い換え", clean=True, emb="large", hybrid=True, rerank=True, rewrite=True),
}

def retrieve(q, cfg, n=10):
    qq = clean(q) if cfg["clean"] else q
    queries = [qq] + (rewrite(q) if cfg["rewrite"] else [])
    lists = []
    for x in queries:
        lists.append(dense(x, cfg["emb"], 30))
        if cfg["hybrid"]:
            lists.append(sparse(x, 30))
    cand = rrf(lists) if len(lists) > 1 else lists[0]
    if cfg["rerank"]:
        cand = rerank(qq, cand[:30])
    return cand[:n]

for key in "AF":
    print(key, "→", [chunks[i][0].split("/")[-1] for i in retrieve("colab-oss-lab のたとえで、CU とは何ですか。", CONFIGS[key], 5)])
print("言い換えの例:", rewrite("colab-oss-lab のたとえで、CU とは何ですか。"))

## 4. 評価の準備（質問と採点は 08 と同じ）

In [ ]:
SYSTEM_RAG = ("あなたは colab-oss-lab（Google Colab で OSS の AI モデルを試すリポジトリ）の案内役です。"
              "下の【資料】だけを根拠に、日本語で短く答えてください。"
              "資料に答えが書かれていなければ、推測せずに「資料に記載がありません」と答えてください。"
              "最後に（出典: ファイル名）を付けてください。")

def answer(q, idx):
    docs = "\n\n".join(f"[{chunks[i][0]} / {chunks[i][1]}]\n{chunks[i][2]}" for i in idx)
    r = client.chat.completions.create(model="gemma4-26b", max_tokens=256, temperature=0, extra_body=NO_THINK,
        messages=[{"role": "system", "content": SYSTEM_RAG}, {"role": "user", "content": f"【資料】\n{docs}\n\n【質問】\n{q}"}])
    return r.choices[0].message.content.strip()

facts = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/moruku36/colab-oss-lab/main/data/06_qlora_facts.json"))["facts"]
def hit(text, keyword_sets):
    t = text.replace(",", "").replace("，", "")
    return any(all(k in t for k in ks) for ks in keyword_sets)
# 08 で「意味は合っているのに × になった」2 問の言葉を足した採点（参考）
EXTRA = {"exp03_acc": [["変わらず"]], "exp05_3bit": [["崩れ"]]}

UNANSWERABLE = [
    "colab-oss-lab で Llama 3.3 70B を動かしたときの速さは何トークン/秒でしたか。",
    "colab-oss-lab の実験で、A100 を使ったときの Gemma 4 31B の速さはいくつでしたか。",
    "colab-oss-lab の実験にかかった電気代はいくらですか。",
    "colab-oss-lab の実験12の結果を教えてください。",
    "colab-oss-lab で TPU を使ったときの結果はどうでしたか。",
]
REFUSE = ["記載がありません", "記載されていません", "見つかりません", "分かりません", "わかりません", "書かれていません", "載っていません"]

# 正解が書かれた段落（検索の採点用）
gold = {f["id"]: [i for i, t in enumerate(texts) if hit(chunks[i][2], f["keywords"])] for f in facts}
print("正解の言葉を含む段落が無い質問:", [k for k, v in gold.items() if not v])

## 5. 6 つの設定を比べる

In [ ]:
from concurrent.futures import ThreadPoolExecutor
results, per_q = {}, {}
for key, cfg in CONFIGS.items():
    t = time.time()
    got = [retrieve(f["test"], cfg, 10) for f in facts]
    ret_sec = (time.time() - t) / len(facts)
    ranks = []
    for f, g in zip(facts, got):
        r = [k for k, i in enumerate(g) if hit(chunks[i][2], f["keywords"])]
        ranks.append(r[0] + 1 if r else None)
    with ThreadPoolExecutor(max_workers=16) as ex:
        ans = list(ex.map(lambda x: answer(x[0]["test"], x[1][:5]), zip(facts, got)))
        un_ctx = [retrieve(q, cfg, 5) for q in UNANSWERABLE]
        un_ans = list(ex.map(lambda x: answer(*x), zip(UNANSWERABLE, un_ctx)))
    ok = [hit(a, f["keywords"]) for a, f in zip(ans, facts)]
    ok2 = [hit(a, f["keywords"] + EXTRA.get(f["id"], [])) for a, f in zip(ans, facts)]
    results[key] = dict(
        name=cfg["name"], ret_sec=ret_sec,
        hit1=np.mean([r is not None and r <= 1 for r in ranks]),
        hit5=np.mean([r is not None and r <= 5 for r in ranks]),
        hit10=np.mean([r is not None for r in ranks]),
        mrr=np.mean([1 / r if r else 0 for r in ranks]),
        acc=np.mean(ok), acc2=np.mean(ok2),
        refuse=sum(any(w in a for w in REFUSE) for a in un_ans),
        readme=np.mean([sum(chunks[i][0] == "README.md" for i in g[:5]) for g in got]))
    per_q[key] = [dict(id=f["id"], rank=r, ok=o, answer=a, top=[chunks[i][0].split("/")[-1] for i in g[:5]])
                  for f, r, o, a, g in zip(facts, ranks, ok, ans, got)]
    per_q[key + "_un"] = [dict(q=q, answer=a) for q, a in zip(UNANSWERABLE, un_ans)]
    r = results[key]
    print(f"{key} {r['name']:<28} 検索@5 {r['hit5']:.0%}  @1 {r['hit1']:.0%}  MRR {r['mrr']:.2f}  "
          f"正答 {r['acc']:.0%}（広げた採点 {r['acc2']:.0%}）  断る {r['refuse']}/5  README {r['readme']:.1f}/5  検索 {r['ret_sec']:.2f}秒")

## 6. まとめて、実行記録を出す

In [ ]:
from datetime import datetime, timezone, timedelta
vv = subprocess.check_output(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"], text=True).strip()
best = max(results, key=lambda k: (results[k]["acc"], results[k]["hit5"], -ord(k)))
b = results[best]
checks = {
    f"検索が正解の段落を拾えた割合（上位 5）が 90% 以上（{best}）": b["hit5"] >= 0.90,
    f"RAG の正答率が 80% 以上（{best}）": b["acc"] >= 0.80,
    f"資料にない質問 5 問のうち 4 問以上で「記載がありません」（{best}）": b["refuse"] >= 4,
    f"1 問あたりの検索が 1 秒以内（{best}）": b["ret_sec"] <= 1.0,
}
ok = all(checks.values())
now = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")
L = ["# 実行記録: 10 RAG の検索をよくする", "",
     f"- 実行日: {now}", "- 実行場所: Google Colab",
     f"- GPU: {gpu_name} / VRAM {round(vram_total_gb, 1)} GB",
     f"- モデル: {MODEL}（vLLM {vv} の OpenAI 互換 API、thinking オフ、temperature 0、gpu-memory-utilization 0.90）",
     f"- サーバー起動: {server_start_min:.1f} 分 / KV キャッシュ: {kv[-1] if kv else '-'} トークン",
     f"- 資料: コミット e4570f0 の {len(FILES)} ファイル → {len(chunks)} 段落（08 と同じ）/ 答えに渡すのは上位 5 段落",
     "- 検索の部品: intfloat/multilingual-e5-small（CPU）/ intfloat/multilingual-e5-large（CPU）/ BM25（日本語は 2 文字区切り）/ BAAI/bge-reranker-v2-m3（" + rerank_device + "）",
     f"- いちばん良い設定: **{best}（{b['name']}）**",
     f"- 想定どおりか: {'はい' if ok else 'いいえ'}", "",
     "## まとめ", "",
     "| 設定 | 検索 @1 | 検索 @5 | 検索 @10 | MRR | 正答率 | 正答率（広げた採点） | 資料にない質問で断る | 上位 5 の README | 検索の時間 / 問 |",
     "|---|---|---|---|---|---|---|---|---|---|"]
for k, r in results.items():
    L.append(f"| {k}: {r['name']} | {r['hit1']:.0%} | {r['hit5']:.0%} | {r['hit10']:.0%} | {r['mrr']:.2f} | {r['acc']:.0%} | {r['acc2']:.0%} | {r['refuse']} / 5 | {r['readme']:.1f} | {r['ret_sec']:.2f} 秒 |")
L += ["", "## 判定", ""] + [f"- [{'x' if v else ' '}] {k}" for k, v in checks.items()]
L += ["", f"## 問題ごと（正解の段落の順位 / 正誤）", "", "| id | " + " | ".join(results) + " |", "|---|" + "---|" * len(results)]
for j, f in enumerate(facts):
    L.append(f"| {f['id']} | " + " | ".join(f"{per_q[k][j]['rank'] or '-'} {'○' if per_q[k][j]['ok'] else '×'}" for k in results) + " |")
L += ["", f"## {best} の答え（A で × だった問題）", ""]
for a, x in zip(per_q["A"], per_q[best]):
    if not a["ok"]:
        L += [f"- {x['id']}: A「{a['answer'][:70]}」→ {best}「{x['answer'][:90]}」".replace("\n", " ")]
L += ["", f"## {best} で × の問題", ""]
for x in per_q[best]:
    if not x["ok"]:
        L += [f"- {x['id']}（順位 {x['rank'] or '10 位以内になし'} / 上位: {', '.join(x['top'])}）: {x['answer'][:120]}".replace("\n", " ")]
L += ["", f"## 資料にない質問（{best}）", ""] + [f"- {u['q']} → {u['answer'][:80]}".replace("\n", " ") for u in per_q[best + "_un"]]
L += ["", "## 質問の言い換えの例（F）", ""] + [f"- {q} → {' / '.join(v)}" for q, v in list(_rw.items())[:6]]
print("\n".join(L))
json.dump(dict(results=results, per_q=per_q, rewrites=_rw), open("/content/rag10_result.json", "w"), ensure_ascii=False, indent=1, default=float)

## 7. サーバーを止める

終わったら vLLM のサーバーを止めて、**ランタイム → セッションを管理 → 解放** を押してください。

In [ ]:
server.terminate()
server.wait(timeout=60)
print("サーバーを止めました")